In [3]:
"""
MVN outlier 偵測 — build → fit → check
3D 資料,測試兩個新點是否在 95% 信賴橢球內
"""
import numpy as np
from scipy.stats import chi2
import matplotlib.pyplot as plt

# ============================================================
# 計算區
# ============================================================

# ---- STEP 1: BUILD (用訓練資料學 mu 跟 Sigma) ----
rng = np.random.default_rng(42)
d = 3  # 變數個數

# 假裝這是你的訓練資料
true_mu = np.array([0, 0, 0])
true_Sigma = np.array([[2.0,  0.5, -0.3],
                       [0.5,  1.5,  0.4],
                       [-0.3, 0.4,  1.0]])
X_train = rng.multivariate_normal(true_mu, true_Sigma, size=200)

# 從資料估參數 (這就是「build」)
mu_hat = X_train.mean(axis=0)
S = np.cov(X_train, rowvar=False)
S_inv = np.linalg.inv(S)

print("=" * 50)
print("STEP 1: BUILD — 從訓練資料估參數")
print("=" * 50)
print(f"估計 mu = {mu_hat}")
print(f"估計 Sigma =\n{S}")


# ---- STEP 2: 算 95% 門檻 ----
c = chi2.ppf(0.95, df=d)

print(f"\n95% 門檻: c = chi^2({d}, 0.95) = {c:.4f}")
print(f"規則: d^2 <= {c:.4f} 算內,> {c:.4f} 算外")


# ---- STEP 3: FIT 新點 (算 Mahalanobis d^2) ----
def mahalanobis_sq(x, mu, S_inv):
    """單一新點的 d^2"""
    diff = x - mu
    return diff @ S_inv @ diff

# 兩個測試點
x_A = np.array([0.5, 0.3, 0.2])   # 靠近 mean
x_B = np.array([3.0, 2.5, 2.0])   # 離 mean 遠

d2_A = mahalanobis_sq(x_A, mu_hat, S_inv)
d2_B = mahalanobis_sq(x_B, mu_hat, S_inv)

print("\n" + "=" * 50)
print("STEP 3: FIT — 算新點的 Mahalanobis d^2")
print("=" * 50)
print(f"Point A = {x_A}: d^2 = {d2_A:.3f}")
print(f"Point B = {x_B}: d^2 = {d2_B:.3f}")


# ---- STEP 4: CHECK (跟門檻比較 + 機率解讀) ----
verdict_A = "INSIDE (95% 內)" if d2_A <= c else "OUTSIDE (outlier)"
verdict_B = "INSIDE (95% 內)" if d2_B <= c else "OUTSIDE (outlier)"

# Tail probability: 「看到至少這麼極端的機率」
p_A = 1 - chi2.cdf(d2_A, df=d)
p_B = 1 - chi2.cdf(d2_B, df=d)

print("\n" + "=" * 50)
print("STEP 4: CHECK — 跟門檻比較")
print("=" * 50)
print(f"Point A: d^2={d2_A:.3f} vs c={c:.3f} → {verdict_A}")
print(f"         至少這麼極端的機率 = {p_A*100:.2f}%  (高 → 典型)")
print(f"\nPoint B: d^2={d2_B:.3f} vs c={c:.3f} → {verdict_B}")
print(f"         至少這麼極端的機率 = {p_B*100:.4f}%  (低 → 罕見)")


# ============================================================
# 視覺化區 (全部畫圖放在最下面)
# ============================================================
fig = plt.figure(figsize=(13, 6))

# Left: 3D scatter
ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(X_train[:, 0], X_train[:, 1], X_train[:, 2],
            alpha=0.3, s=15, color='steelblue', label=f'Training (n={len(X_train)})')
ax1.scatter(*mu_hat, color='black', s=150, marker='X', label='mu_hat')
ax1.scatter(*x_A, color='green', s=250, marker='o', edgecolor='black',
            linewidth=2, label=f'A: d^2={d2_A:.2f} (inside)')
ax1.scatter(*x_B, color='red', s=250, marker='o', edgecolor='black',
            linewidth=2, label=f'B: d^2={d2_B:.2f} (outside)')
ax1.set_xlabel('X1'); ax1.set_ylabel('X2'); ax1.set_zlabel('X3')
ax1.set_title('3D training data + test points')
ax1.legend(loc='upper left', fontsize=9)

# Right: bar chart 比較 d^2 跟門檻
ax2 = fig.add_subplot(122)
labels = ['Point A\n(inside)', 'Point B\n(outside)']
values = [d2_A, d2_B]
colors = ['green', 'red']
bars = ax2.bar(labels, values, color=colors, alpha=0.7, edgecolor='black')
ax2.axhline(c, color='black', linestyle='--', lw=2,
            label=f'95% threshold\nc = chi^2({d}, 0.95) = {c:.2f}')
for bar, val in zip(bars, values):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.4,
             f'd^2 = {val:.2f}', ha='center', fontweight='bold')
ax2.set_ylabel('Mahalanobis d^2')
ax2.set_title('d^2 vs threshold')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/Users/fangsiyu/Desktop/sdu-2026-code/805_multivariate_statistical_analysis/note/fig6_outlier_3d.png', dpi=110, bbox_inches='tight')
plt.close()
print("\n→ 圖存到 fig_outlier_3d.png")

STEP 1: BUILD — 從訓練資料估參數
估計 mu = [-0.02882529  0.06754468  0.06113568]
估計 Sigma =
[[ 2.02184064  0.54082694 -0.1825178 ]
 [ 0.54082694  1.5190744   0.4442951 ]
 [-0.1825178   0.4442951   0.85155106]]

95% 門檻: c = chi^2(3, 0.95) = 7.8147
規則: d^2 <= 7.8147 算內,> 7.8147 算外

STEP 3: FIT — 算新點的 Mahalanobis d^2
Point A = [0.5 0.3 0.2]: d^2 = 0.180
Point B = [3.  2.5 2. ]: d^2 = 10.490

STEP 4: CHECK — 跟門檻比較
Point A: d^2=0.180 vs c=7.815 → INSIDE (95% 內)
         至少這麼極端的機率 = 98.07%  (高 → 典型)

Point B: d^2=10.490 vs c=7.815 → OUTSIDE (outlier)
         至少這麼極端的機率 = 1.4827%  (低 → 罕見)

→ 圖存到 fig_outlier_3d.png


In [2]:
"""
從 chi²(d) 的定義,一路推到 Pr(X ∈ E_c) = Pr(chi²(d) ≤ c)
6 步漸進,每步只加一點點
"""
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)
n = 50_000

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# ============================================================
# STEP 1: chi²(1) 的定義就是 Z²,Z ~ N(0,1)
# ============================================================
Z = rng.normal(0, 1, n)
Z_sq = Z**2  # 一個 standard normal 的平方

ax = axes[0, 0]
ax.hist(Z_sq, bins=80, density=True, alpha=0.6, range=(0, 10), color='steelblue')
x = np.linspace(0.01, 10, 200)
ax.plot(x, stats.chi2.pdf(x, df=1), 'r-', lw=2, label='chi^2(1)')
ax.set_title('Step 1: Z^2 where Z~N(0,1)\n=> chi^2(1) BY DEFINITION')
ax.set_xlabel('Z^2'); ax.legend(); ax.set_xlim(0, 10)

# ============================================================
# STEP 2: chi²(d) 的定義就是 d 個獨立 Z² 相加
# ============================================================
Z1 = rng.normal(0, 1, n)
Z2 = rng.normal(0, 1, n)
sum_sq = Z1**2 + Z2**2  # 兩個獨立 N(0,1) 平方相加

ax = axes[0, 1]
ax.hist(sum_sq, bins=80, density=True, alpha=0.6, range=(0, 15), color='steelblue')
x = np.linspace(0.01, 15, 200)
ax.plot(x, stats.chi2.pdf(x, df=2), 'r-', lw=2, label='chi^2(2)')
ax.set_title('Step 2: Z1^2 + Z2^2\n=> chi^2(2) BY DEFINITION')
ax.set_xlabel('sum of squares'); ax.legend(); ax.set_xlim(0, 15)

# 重點: chi²(d) 就是「d 個獨立 standard normal 平方的和」,沒別的

# ============================================================
# STEP 3: c = chi²_{d, 0.95} 就是 chi² 分布的 95% 分位點
# ============================================================
c_95 = stats.chi2.ppf(0.95, df=2)  # ≈ 5.99
print(f"c = chi^2_{{2, 0.95}} = {c_95:.4f}")
print(f"意思: chi^2(2) 有 95% 機率落在 {c_95:.4f} 以下")

ax = axes[0, 2]
ax.hist(sum_sq, bins=80, density=True, alpha=0.6, range=(0, 15), color='steelblue')
ax.plot(x, stats.chi2.pdf(x, df=2), 'r-', lw=2)
# 把 95% 區域塗綠
x_left = np.linspace(0.01, c_95, 100)
ax.fill_between(x_left, stats.chi2.pdf(x_left, df=2), alpha=0.3, color='green',
                label=f'95% area\n(Pr <= c)')
ax.axvline(c_95, color='black', linestyle='--', lw=2, label=f'c = {c_95:.2f}')
ax.set_title(f'Step 3: c is just "the 95% percentile"\nof chi^2(2)')
ax.set_xlabel('chi^2(2) value'); ax.legend(); ax.set_xlim(0, 15)

# 數值驗證
frac = (sum_sq <= c_95).mean()
print(f"驗證: 模擬出 {frac*100:.2f}% 的 sum_sq 落在 c 以下 (理論 95%)\n")

# ============================================================
# STEP 4: 關鍵連結 — MVN 的 Mahalanobis d² 也服從 chi²(d)
# ============================================================
# 為什麼? 因為 d² = ||Sigma^(-1/2) (X - mu)||² = ||Z||² 而 Z ~ N(0, I)
# 所以 d² = Z1² + Z2² + ... = chi²(d) by definition

mu = np.array([0, 0])
Sigma = np.array([[2.0, 1.0],
                  [1.0, 1.5]])  # 有相關性,不是 identity
X_mvn = rng.multivariate_normal(mu, Sigma, size=n)

Sigma_inv = np.linalg.inv(Sigma)
Xc = X_mvn - mu
d_sq = np.einsum('ni,ij,nj->n', Xc, Sigma_inv, Xc)  # Mahalanobis d²

ax = axes[1, 0]
ax.hist(d_sq, bins=80, density=True, alpha=0.6, range=(0, 15), color='orange')
ax.plot(x, stats.chi2.pdf(x, df=2), 'r-', lw=2, label='chi^2(2)')
ax.set_title('Step 4: Mahalanobis d^2 for MVN data\n=> SAME chi^2(2)! (this is the magic)')
ax.set_xlabel('d^2 = (x-mu)^T Sigma^-1 (x-mu)'); ax.legend(); ax.set_xlim(0, 15)

# ============================================================
# STEP 5: 用 c 切資料 — 95% 的點會在橢圓內
# ============================================================
inside = d_sq <= c_95
print(f"Step 5: 用 c = {c_95:.4f} 切資料")
print(f"  {inside.sum()} / {n} = {inside.mean()*100:.2f}% 的點滿足 d^2 <= c (理論 95%)")

# 用小一點的 sample 畫圖 (清楚一點)
idx = rng.choice(n, 2000, replace=False)
X_plot = X_mvn[idx]
inside_plot = inside[idx]

ax = axes[1, 1]
ax.scatter(X_plot[~inside_plot, 0], X_plot[~inside_plot, 1], 
           s=10, alpha=0.5, color='red', label=f'outside ({(~inside_plot).sum()})')
ax.scatter(X_plot[inside_plot, 0], X_plot[inside_plot, 1], 
           s=10, alpha=0.5, color='green', label=f'inside ({inside_plot.sum()})')

# 畫 95% 橢圓: (x-mu)^T Sigma^-1 (x-mu) = c
theta = np.linspace(0, 2*np.pi, 200)
eigvals, eigvecs = np.linalg.eigh(Sigma)
circle = np.sqrt(c_95) * np.column_stack([np.cos(theta), np.sin(theta)])
ellipse = circle @ np.diag(np.sqrt(eigvals)) @ eigvecs.T + mu
ax.plot(ellipse[:, 0], ellipse[:, 1], 'k-', lw=2.5, label='95% ellipse (d^2 = c)')

ax.set_title('Step 5: E_c = {x : d^2(x) <= c}\n= 95% probability ellipsoid')
ax.set_xlabel('x_1'); ax.set_ylabel('x_2')
ax.legend(loc='upper right'); ax.set_aspect('equal'); ax.grid(True, alpha=0.3)

# ============================================================
# STEP 6: 驗證 Pr(X ∈ E_c) = Pr(chi²(d) ≤ c)
# ============================================================
ax = axes[1, 2]
ax.axis('off')
ax.text(0.05, 0.5, f"""
THE BIG EQUATION

  Pr(X in E_c)  =  Pr(chi^2(d) <= c)
       ^                 ^
    (left side)       (right side)
   from your data    from chi^2 table


For d=2, c = chi^2_{{2, 0.95}} = {c_95:.4f}:

  Left side  (from MVN samples):
    {inside.mean()*100:.2f}% of {n} points inside E_c

  Right side (from chi^2 definition):
    Pr(chi^2(2) <= {c_95:.2f}) = 95.00%

  They match! ✓

  This is because d^2 ~ chi^2(d)
  (Step 4's magic)
""",
fontsize=10, verticalalignment='center', family='monospace')
ax.set_title('Step 6: Why it all works')

plt.tight_layout()
plt.savefig('/Users/fangsiyu/Desktop/sdu-2026-code/805_multivariate_statistical_analysis/note/fig6_chi2_progression.png', dpi=110, bbox_inches='tight')
plt.close()
print("\n→ 圖存到 chi2_progression.png")

c = chi^2_{2, 0.95} = 5.9915
意思: chi^2(2) 有 95% 機率落在 5.9915 以下
驗證: 模擬出 94.89% 的 sum_sq 落在 c 以下 (理論 95%)

Step 5: 用 c = 5.9915 切資料
  47509 / 50000 = 95.02% 的點滿足 d^2 <= c (理論 95%)

→ 圖存到 chi2_progression.png
